# Aneks techniczny: implementacja

Plik zawiera pełną implementację procedury badawczej: pobieranie danych, konstrukcję zmiennych
objaśniających, walidację kroczącą, reguły inwestycyjne, symulację oraz testy statystyczne.

Wyniki i ich omówienie znajdują się w pliku `wyniki_obliczen.ipynb`, który ładuje ten plik
na starcie. Tutaj nie są wykonywane żadne obliczenia — wyłącznie definicje.

## Spis treści

1. Parametry badania
2. Pobieranie danych
3. Zmienne objaśniające
4. Zmienne objaśniane
5. Walidacja krocząca
6. Modele i dobór hiperparametrów
7. Pętla ucząca
8. Reguły inwestycyjne
9. Symulacja i metryki
10. Testy statystyczne
11. Analiza portfelowa i ważność zmiennych
12. Przygotowanie danych


## 1. Parametry badania


In [ ]:
import warnings, os, json, time, itertools, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score, brier_score_loss)
from sklearn.inspection import permutation_importance

import yfinance as yf
from xgboost import XGBClassifier

OPT = {}
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    import tensorflow as tf
    OPT['tensorflow'] = True
except Exception:
    OPT['tensorflow'] = False
try:
    import requests
    OPT['requests'] = True
except Exception:
    OPT['requests'] = False
try:
    from statsmodels.tsa.stattools import adfuller
    OPT['statsmodels'] = True
except Exception:
    OPT['statsmodels'] = False

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


In [ ]:
TICKERS = {
    'BTC-USD': 'Bitcoin',
    'SPY':     'S&P 500 (akcje)',
    'TLT':     'Obligacje skarbowe USA 20+ lat',
}
TICKER_WERYFIKACYJNY = 'ETH-USD'

START_DATE = pd.Timestamp('2017-01-01')
END_DATE   = pd.Timestamp.today().normalize()
OOS_START  = pd.Timestamp('2021-01-01')

WF_MODE, ROLLING_TRAIN = 'expanding', 750
TEST_WINDOW, STEP = 42, 42
PURGE, EMBARGO = 50, 5
VAL_FRACTION = 0.20

DO_TUNING, TUNE_EVERY_N_STEPS, TUNE_METRIC, TUNE_LSTM = True, 3, 'auc', False
SEQUENCE_LENGTH, LSTM_N_SEEDS, LSTM_EPOCHS = 20, 3, 30
CORR_PRUNE_THRESHOLD = 0.95
SAMPLE_WEIGHT_MODE = 'abs_return'

ZADANIA_GLOWNE = ['direction_1d', 'no_large_drop']
ZADANIA_ANEKS  = ['direction_5d', 'triple_barrier']
HORIZON_5D, TB_HORIZON, TB_SIGMA_MULT, LARGE_DROP_SIGMA = 5, 10, 1.0, 1.5

TRANSACTION_COST = 0.001
COST_GRID = [0.0, 0.0005, 0.001, 0.0025, 0.005]
EXECUTION = 'close_to_close'
USE_RISK_FREE = True
SLIPPAGE_STOP = 0.0015
MAX_LEVERAGE = 1.0

TOP_N_KROCZACY = 10

STRATEGIA_PROSTA = dict(mode='binary', threshold=0.50)
STRATEGIA_GLOWNA = dict(mode='binary', threshold=0.55, exit_threshold=0.45,
                        min_hold=5, vol_target=0.30)

WARIANTY_ANEKS = {
    'próg 0,50':                dict(mode='binary', threshold=0.50),
    'próg 0,55':                dict(mode='binary', threshold=0.55),
    'próg dobierany kroczaco':  dict(mode='binary', threshold='tuned'),
    'skalowanie pozycji':       dict(mode='scaled', threshold=0.50, k=4.0),
    'kontrola zmiennosci':      dict(mode='binary', threshold=0.50, vol_target=0.30),
    'filtr SMA200':             dict(mode='binary', threshold=0.50, sma_filter=200),
    'stop-loss 7%':             dict(mode='binary', threshold=0.50, stop_loss=0.07),
    'konfiguracja glowna':      STRATEGIA_GLOWNA,
}

FREQ_MAP = {'BTC-USD': 365, 'SPY': 252, 'TLT': 252, 'ETH-USD': 365}
MIN_YEARS_FOR_CAGR = 0.75
N_BOOTSTRAP, MEAN_BLOCK, N_RANDOM = 1000, 20, 500
RANDOM_STATE = 42

MODELS = ['LR', 'XGB', 'LSTM']
MODEL_LABEL = {'LR': 'Regresja logistyczna', 'XGB': 'XGBoost', 'LSTM': 'LSTM'}
NAZWA_ZADANIA = {
    'direction_1d':   'kierunek zmiany (D+1)',
    'no_large_drop':  'ryzyko duzego spadku',
    'direction_5d':   'kierunek zmiany (D+5)',
    'triple_barrier': 'bariery zmiennosciowe',
}
OKRESY = {
    'Hossa 2021':         ('2021-01-01', '2021-12-31'),
    'Bessa 2022':         ('2022-01-01', '2022-12-31'),
    'Odbicie 2023':       ('2023-01-01', '2023-12-31'),
    'Hossa 2024':         ('2024-01-01', '2024-12-31'),
    '2025-2026':          ('2025-01-01', None),
}

CACHE_DIR, WYNIKI_DIR = 'cache', 'wyniki_final'
SCIEZKI_CACHE = {
    'glowne':     os.path.join(CACHE_DIR, 'wf_results.pkl'),
    'odpornosc':  os.path.join(CACHE_DIR, 'test3_robustness.pkl'),
    'eth':        os.path.join(CACHE_DIR, 'test6_eth.pkl'),
    'lstm10':     os.path.join(CACHE_DIR, 'test7_lstm10.pkl'),
}
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(WYNIKI_DIR, exist_ok=True)


## 2. Pobieranie danych


In [ ]:
def _cache_path(name):
    return os.path.join(CACHE_DIR, f"{name.replace('/', '_').replace('^', 'idx_')}.pkl")


def download_yf(ticker, start=None, end=None, use_cache=True):
    start = START_DATE if start is None else start
    end = END_DATE if end is None else end
    p = _cache_path(f'yf_{ticker}')
    if use_cache and os.path.exists(p):
        try:
            df = pd.read_pickle(p)
            if df.index.max() >= pd.Timestamp(end) - pd.Timedelta(days=5):
                return df
        except Exception:
            pass
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    if df is None or df.empty:
        raise ValueError(f'brak danych: {ticker}')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[[c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in df.columns]].copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df.sort_index()
    df.to_pickle(p)
    return df


EXOG_SPEC = {
    '^VIX': dict(kind='level', name='VIX'),      '^VIX3M': dict(kind='level', name='VIX3M'),
    '^TNX': dict(kind='level', name='UST10Y'),   '^FVX':   dict(kind='level', name='UST5Y'),
    '^IRX': dict(kind='level', name='UST13W'),   'HYG':    dict(kind='price', name='HYG'),
    'LQD':  dict(kind='price', name='LQD'),      'UUP':    dict(kind='price', name='USD'),
    'DX-Y.NYB': dict(kind='price', name='DXY'),  'GLD':    dict(kind='price', name='GOLD'),
    'SPY':  dict(kind='price', name='SPY'),      'QQQ':    dict(kind='price', name='QQQ'),
    'TLT':  dict(kind='price', name='TLT'),      'ETH-USD': dict(kind='price', name='ETH'),
}

BLOCKCHAIN_CHARTS = {
    'AdrActCnt': 'n-unique-addresses', 'TxTfrValAdjUSD': 'estimated-transaction-volume-usd',
    'HashRate': 'hash-rate', 'CapMrktCurUSD': 'market-cap',
    'NTx': 'n-transactions', 'FeesUSD': 'transaction-fees-usd',
}


def fetch_fear_greed():
    p = _cache_path('fng')
    if os.path.exists(p):
        return pd.read_pickle(p)
    js = requests.get('https://api.alternative.me/fng/?limit=0&format=json',
                      timeout=30).json()['data']
    s = pd.Series({pd.to_datetime(int(x['timestamp']), unit='s').normalize(): float(x['value'])
                   for x in js}, name='FNG').sort_index()
    s.to_pickle(p)
    return s


def fetch_onchain(metrics=tuple(BLOCKCHAIN_CHARTS)):
    p = _cache_path('coinmetrics')
    if os.path.exists(p):
        return pd.read_pickle(p)
    out = {}
    for name in metrics:
        try:
            vals = requests.get(f'https://api.blockchain.info/charts/{BLOCKCHAIN_CHARTS[name]}',
                                params=dict(timespan='all', format='json'),
                                timeout=45).json().get('values', [])
            if vals:
                out[name] = pd.Series({pd.to_datetime(v['x'], unit='s').normalize(): float(v['y'])
                                       for v in vals}).sort_index()
        except Exception:
            pass
        time.sleep(0.3)
    if not out:
        raise RuntimeError('brak danych sieciowych')
    df = pd.DataFrame(out).sort_index()
    if {'CapMrktCurUSD', 'TxTfrValAdjUSD'}.issubset(df.columns):
        df['NVT'] = df['CapMrktCurUSD'] / df['TxTfrValAdjUSD'].replace(0, np.nan)
    df = df[df.index >= START_DATE]
    df.to_pickle(p)
    return df


def fetch_funding(symbol='BTCUSDT'):
    p = _cache_path(f'funding_{symbol}')
    if os.path.exists(p):
        return pd.read_pickle(p)
    out, start_ms = [], int(pd.Timestamp('2019-09-01').timestamp() * 1000)
    end_ms = int(END_DATE.timestamp() * 1000)
    while start_ms < end_ms:
        js = requests.get('https://fapi.binance.com/fapi/v1/fundingRate',
                          params=dict(symbol=symbol, startTime=start_ms, limit=1000),
                          timeout=30).json()
        if not isinstance(js, list) or not js:
            break
        out.extend(js)
        last = int(js[-1]['fundingTime'])
        if last <= start_ms:
            break
        start_ms = last + 1
        time.sleep(0.15)
    df = pd.DataFrame(out)
    if df.empty:
        raise RuntimeError('brak danych')
    df['t'] = pd.to_datetime(df['fundingTime'], unit='ms').dt.normalize()
    s = df.groupby('t')['fundingRate'].apply(lambda x: pd.to_numeric(x).sum()).rename('FUNDING')
    s.to_pickle(p)
    return s


def rf_daily_for(index, freq):
    if not USE_RISK_FREE or rf_annual.empty:
        return pd.Series(0.0, index=index)
    s = rf_annual.reindex(rf_annual.index.union(index)).ffill().reindex(index)
    return (s / freq).fillna(0.0)


## 3. Zmienne objaśniające

Zmienne techniczne konstruowane są w postaci bezwymiarowej — jako odchylenia relatywne, a nie
wartości nominalne. MACD dzielony jest przez cenę: w postaci nominalnej jego wariancja rośnie
proporcjonalnie do poziomu ceny, co powoduje, że standaryzacja wyznaczona na zbiorze treningowym
nie odpowiada rozkładowi w okresie testowym.

Zmienne opisujące poziomy stóp procentowych i spreadów kredytowych występują wyłącznie
w postaciach różnicowych — w badanym okresie ich poziomy nie spełniają warunku stacjonarności.

Szeregi zewnętrzne dopasowywane są do kalendarza instrumentu wyłącznie przez przeniesienie
ostatniej znanej wartości w przód. Dla modelu na SPY usuwane są zmienne pochodzące z SPY,
analogicznie dla TLT i ETH.


In [ ]:
ZSCORE_WINDOW = {'UST13W': 126}
ZSCORE_DEFAULT = 252


def add_technical_features(df, freq):
    d = df.copy()
    c = d['Close']
    h, l, v = d.get('High', c), d.get('Low', c), d.get('Volume')

    d['log_ret_1'] = np.log(c / c.shift(1))
    for lag in [2, 3, 5, 10, 20]:
        d[f'log_ret_lag{lag}'] = d['log_ret_1'].rolling(lag).sum()
    for w in [5, 10, 20, 50, 200]:
        d[f'price_sma_{w}_ratio'] = c / c.rolling(w).mean() - 1
    d['sma_50_200_ratio'] = c.rolling(50).mean() / c.rolling(200).mean() - 1

    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    d['macd_norm'] = macd / c
    d['macd_hist_norm'] = (macd - macd.ewm(span=9, adjust=False).mean()) / c

    delta = c.diff()
    ag = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    al = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    d['rsi_14'] = (100 - 100 / (1 + ag / al.replace(0, np.nan))) / 100.0

    bb_mid, bb_std = c.rolling(20).mean(), c.rolling(20).std()
    d['bb_pctb'] = (c - (bb_mid - 2 * bb_std)) / (4 * bb_std)
    d['bb_width'] = (4 * bb_std) / bb_mid

    for w in [5, 10, 20, 50]:
        d[f'volatility_{w}'] = d['log_ret_1'].rolling(w).std()
    d['vol_ratio_10_50'] = d['volatility_10'] / d['volatility_50']
    d['ret_skew_20'] = d['log_ret_1'].rolling(20).skew()
    d['ret_kurt_20'] = d['log_ret_1'].rolling(20).kurt()

    tr = pd.concat([(h - l), (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
    d['atr_14_norm'] = tr.rolling(14).mean() / c
    d['drawdown_252'] = c / c.rolling(252, min_periods=60).max() - 1

    if v is not None and v.notna().any() and v.abs().sum() > 0:
        d['volume_zscore'] = (v - v.rolling(20).mean()) / v.rolling(20).std()
        d['volume_trend'] = v.rolling(5).mean() / v.rolling(50).mean() - 1
    else:
        d['volume_zscore'] = d['volume_trend'] = 0.0

    d['dow_sin'] = np.sin(2 * np.pi * d.index.dayofweek / 7)
    d['dow_cos'] = np.cos(2 * np.pi * d.index.dayofweek / 7)
    d['month_sin'] = np.sin(2 * np.pi * (d.index.month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (d.index.month - 1) / 12)
    d['ann_vol_20'] = d['volatility_20'] * np.sqrt(freq)
    return d


def build_exog_features(target_index, exclude_tickers=()):
    if exog_levels.empty:
        return pd.DataFrame(index=target_index)
    cols = [c for c in exog_levels.columns if exog_source.get(c) not in exclude_tickers]
    lv = exog_levels[cols].reindex(
        pd.date_range(exog_levels.index.min(),
                      max(exog_levels.index.max(), target_index.max()), freq='D')).ffill()

    out = pd.DataFrame(index=lv.index)
    for c in cols:
        s = lv[c]
        if exog_kinds[c] == 'price':
            r = np.log(s / s.shift(1))
            out[f'x_{c}_ret1'], out[f'x_{c}_ret5'] = r, r.rolling(5).sum()
            out[f'x_{c}_ret20'], out[f'x_{c}_vol20'] = r.rolling(20).sum(), r.rolling(20).std()
        else:
            w = ZSCORE_WINDOW.get(c, ZSCORE_DEFAULT)
            m, sd = s.rolling(w, min_periods=60).mean(), s.rolling(w, min_periods=60).std()
            out[f'x_{c}_lvl_z'] = (s - m) / sd
            out[f'x_{c}_chg1'], out[f'x_{c}_chg5'] = s.diff(), s.diff(5)

    if {'VIX3M', 'VIX'}.issubset(cols):
        out['x_vix_term_structure'] = lv['VIX3M'] / lv['VIX']
    if {'HYG', 'LQD'}.issubset(cols):
        cs = np.log(lv['HYG'] / lv['LQD'])
        out['x_credit_spread_chg5'], out['x_credit_spread_chg20'] = cs.diff(5), cs.diff(20)
    if {'UST10Y', 'UST5Y'}.issubset(cols):
        cv = lv['UST10Y'] - lv['UST5Y']
        out['x_curve_10y_5y_chg5'], out['x_curve_10y_5y_chg20'] = cv.diff(5), cv.diff(20)
    if {'UST10Y', 'UST13W'}.issubset(cols):
        cv = lv['UST10Y'] - lv['UST13W']
        out['x_curve_10y_3m_chg5'], out['x_curve_10y_3m_chg20'] = cv.diff(5), cv.diff(20)
    if {'SPY', 'TLT'}.issubset(cols):
        out['x_stocks_bonds_ratio'] = np.log(lv['SPY'] / lv['TLT']).diff(20)
    if {'QQQ', 'SPY'}.issubset(cols):
        out['x_growth_value_proxy'] = np.log(lv['QQQ'] / lv['SPY']).diff(20)
    return out.reindex(target_index)


def _zscore(s, w, mp):
    return (s - s.rolling(w, min_periods=mp).mean()) / s.rolling(w, min_periods=mp).std()


def build_crypto_features(target_index, funding_key='Funding'):
    out = pd.DataFrame(index=pd.date_range(START_DATE, END_DATE, freq='D'))
    if 'FearGreed' in crypto_extra:
        s = crypto_extra['FearGreed'].reindex(out.index).ffill()
        out['c_fng'], out['c_fng_chg5'] = s / 100.0, (s / 100.0).diff(5)
        out['c_fng_z'] = _zscore(s, 90, 30)
    if 'OnChain' in crypto_extra:
        cm = crypto_extra['OnChain'].reindex(out.index).ffill()
        if 'AdrActCnt' in cm:
            la = np.log(cm['AdrActCnt'].replace(0, np.nan))
            out['c_active_addr_chg30'], out['c_active_addr_z'] = la.diff(30), _zscore(la, 180, 60)
        if 'TxTfrValAdjUSD' in cm:
            out['c_tx_value_chg30'] = np.log(cm['TxTfrValAdjUSD'].replace(0, np.nan)).diff(30)
        if 'NVT' in cm:
            ln = np.log(cm['NVT'].replace(0, np.nan))
            out['c_nvt_z'], out['c_nvt_chg30'] = _zscore(ln, 365, 90), ln.diff(30)
        if 'HashRate' in cm:
            lh = np.log(cm['HashRate'].replace(0, np.nan))
            out['c_hashrate_chg30'], out['c_hashrate_z'] = lh.diff(30), _zscore(lh, 180, 60)
        if {'FeesUSD', 'NTx'}.issubset(cm.columns):
            fee = np.log((cm['FeesUSD'] / cm['NTx'].replace(0, np.nan)).replace(0, np.nan))
            out['c_avg_fee_z'] = _zscore(fee, 180, 60)
    if funding_key in crypto_extra:
        f = crypto_extra[funding_key].reindex(out.index).ffill()
        out['c_funding'], out['c_funding_ma7'] = f, f.rolling(7).mean()
        out['c_funding_z'] = _zscore(f, 90, 30)
    return out.reindex(target_index)


def build_eth_features(target_index):
    out = pd.DataFrame(index=pd.date_range(START_DATE, END_DATE, freq='D'))
    if 'FearGreed' in crypto_extra:
        s = crypto_extra['FearGreed'].reindex(out.index).ffill()
        out['c_fng'], out['c_fng_chg5'] = s / 100.0, (s / 100.0).diff(5)
        out['c_fng_z'] = _zscore(s, 90, 30)
    if 'FundingETH' in crypto_extra:
        f = crypto_extra['FundingETH'].reindex(out.index).ffill()
        out['c_funding'], out['c_funding_ma7'] = f, f.rolling(7).mean()
        out['c_funding_z'] = _zscore(f, 90, 30)
    return out.reindex(target_index)


## 4. Zmienne objaśniane

Cztery sposoby sformułowania zadania prognostycznego. Stopa zwrotu wykorzystywana w symulacji
jest identyczna dla wszystkich — zmienia się wyłącznie to, czego model się uczy.


In [ ]:
def triple_barrier_labels(close, high, low, ann_vol, horizon, sigma_mult, freq):
    n = len(close)
    c = close.values
    hi = high.values if high is not None else c
    lo = low.values if low is not None else c
    sig = (ann_vol / np.sqrt(freq)).values * sigma_mult
    y = np.full(n, np.nan)
    for i in range(n - 1):
        if not np.isfinite(sig[i]) or sig[i] <= 0:
            continue
        up, dn = c[i] * (1 + sig[i]), c[i] * (1 - sig[i])
        end, lab = min(i + horizon, n - 1), np.nan
        for j in range(i + 1, end + 1):
            hu, hd = hi[j] >= up, lo[j] <= dn
            if hu and hd:
                lab = 0.0; break
            if hu:
                lab = 1.0; break
            if hd:
                lab = 0.0; break
        y[i] = lab if np.isfinite(lab) else (1.0 if c[end] > c[i] else 0.0)
    return pd.Series(y, index=close.index)


def add_targets(d, freq):
    c = d['Close']
    h, l = d.get('High', c), d.get('Low', c)
    d['next_return'] = c.shift(-1) / c - 1
    if 'Open' in d.columns:
        o = d['Open']
        d['next_return_oo'] = o.shift(-2) / o.shift(-1) - 1
    else:
        d['next_return_oo'] = d['next_return']
    d['next_low_ret'] = l.shift(-1) / c - 1
    d['next_high_ret'] = h.shift(-1) / c - 1

    d['y_direction_1d'] = (d['next_return'] > 0).astype(float)
    d['y_direction_5d'] = ((c.shift(-HORIZON_5D) / c - 1) > 0).astype(float)
    d['y_triple_barrier'] = triple_barrier_labels(c, h, l, d['ann_vol_20'],
                                                  TB_HORIZON, TB_SIGMA_MULT, freq)
    prog = -(d['ann_vol_20'] / np.sqrt(freq)) * LARGE_DROP_SIGMA
    d['y_no_large_drop'] = (d['next_return'] > prog).astype(float)
    d['w_abs_return'] = d['next_return'].abs()
    return d


TARGET_COLS = {'direction_1d': 'y_direction_1d', 'direction_5d': 'y_direction_5d',
               'triple_barrier': 'y_triple_barrier', 'no_large_drop': 'y_no_large_drop'}
TARGET_HORIZON = {'direction_1d': 1, 'direction_5d': HORIZON_5D,
                  'triple_barrier': TB_HORIZON, 'no_large_drop': 1}
NON_FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume', 'next_return', 'next_return_oo',
                    'next_low_ret', 'next_high_ret', 'ann_vol_20',
                    'w_abs_return'] + list(TARGET_COLS.values())


## 5. Walidacja krocząca

Trzy zabezpieczenia przed przeciekiem informacji: standaryzacja wyznaczana wyłącznie na zbiorze
treningowym, bufor 50 sesji między treningiem a testem (zmienne wykorzystują okna do 252 sesji,
a zmienna objaśniana sięga w przód do H sesji), oraz dobór hiperparametrów na wydzielonym
zbiorze walidacyjnym wewnątrz okna treningowego.


In [ ]:
def walk_forward_splits(index, oos_start, test_window, step, mode='expanding',
                        rolling_train=None, purge=0, embargo=0, min_train=250):
    n = len(index)
    pos = np.where(index >= pd.Timestamp(oos_start))[0]
    if len(pos) == 0:
        return []
    splits, test_start = [], int(pos[0])
    while test_start + test_window <= n:
        train_end = test_start - purge - embargo
        train_start = max(0, train_end - rolling_train) if (mode == 'rolling' and rolling_train) else 0
        if train_end - train_start < min_train:
            test_start += step
            continue
        splits.append((np.arange(train_start, train_end),
                       np.arange(test_start, test_start + test_window)))
        test_start += step
    return splits


def inner_validation_split(train_idx, val_fraction, purge):
    n = len(train_idx)
    n_val = max(30, int(n * val_fraction))
    if n - n_val - purge < 150:
        return train_idx, None
    return train_idx[:max(0, n - n_val - purge)], train_idx[n - n_val:]


def prune_correlated(X_train, threshold=CORR_PRUNE_THRESHOLD):
    if threshold is None or X_train.shape[1] < 2:
        return np.arange(X_train.shape[1])
    C = np.nan_to_num(np.abs(np.corrcoef(np.nan_to_num(X_train), rowvar=False)), nan=0.0)
    keep, dropped = [], set()
    for i in range(C.shape[0]):
        if i in dropped:
            continue
        keep.append(i)
        for j in range(i + 1, C.shape[0]):
            if j not in dropped and C[i, j] > threshold:
                dropped.add(j)
    return np.array(keep)


## 6. Modele i dobór hiperparametrów


In [ ]:
LR_GRID = [dict(C=0.01, class_weight=None), dict(C=0.1, class_weight=None),
           dict(C=1.0, class_weight=None), dict(C=0.1, class_weight='balanced'),
           dict(C=1.0, class_weight='balanced')]

XGB_GRID = [
    dict(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8,
         colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0),
    dict(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8,
         colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0),
    dict(n_estimators=400, max_depth=3, learning_rate=0.02, subsample=0.7,
         colsample_bytree=0.7, min_child_weight=10, reg_lambda=5.0),
    dict(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
         colsample_bytree=0.8, min_child_weight=1, reg_lambda=1.0),
    dict(n_estimators=600, max_depth=2, learning_rate=0.02, subsample=0.7,
         colsample_bytree=0.6, min_child_weight=20, reg_lambda=10.0),
    dict(n_estimators=300, max_depth=5, learning_rate=0.03, subsample=0.9,
         colsample_bytree=0.9, min_child_weight=5, reg_lambda=2.0),
]

LSTM_GRID = [dict(units=32, dropout=0.2, lr=1e-3), dict(units=64, dropout=0.3, lr=1e-3),
             dict(units=16, dropout=0.1, lr=3e-3)]


def fit_logistic(X, y, params, w=None):
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **params))])
    pipe.fit(X, y, clf__sample_weight=w) if w is not None else pipe.fit(X, y)
    return pipe


def fit_xgboost(X, y, params, w=None):
    pos, neg = float(np.sum(y == 1)), float(np.sum(y == 0))
    m = XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0, n_jobs=-1,
                      scale_pos_weight=(neg / pos) if pos > 0 else 1.0, **params)
    m.fit(X, y, sample_weight=w)
    return m


def make_sequences(X, seq_len):
    return np.array([X[i - seq_len:i] for i in range(seq_len, len(X) + 1)])


def fit_predict_lstm(X_tr, y_tr, X_te, params, seed, w_tr=None, seq_len=SEQUENCE_LENGTH):
    if not OPT['tensorflow']:
        return np.full(len(X_te), 0.5)
    tf.keras.utils.set_random_seed(seed)
    sc = StandardScaler()
    Xtr_s, Xte_s = sc.fit_transform(X_tr), sc.transform(X_te)
    Xte_seq = make_sequences(np.vstack([Xtr_s[-(seq_len - 1):], Xte_s]), seq_len)
    Xtr_seq, ytr_seq = make_sequences(Xtr_s, seq_len), y_tr[seq_len - 1:]
    wtr_seq = w_tr[seq_len - 1:] if w_tr is not None else None
    if len(Xtr_seq) < 60:
        return np.full(len(X_te), 0.5)
    model = Sequential([Input(shape=(seq_len, X_tr.shape[1])), LSTM(params['units']),
                        Dropout(params['dropout']), Dense(16, activation='relu'),
                        Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(params['lr']),
                  loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(Xtr_seq, ytr_seq, sample_weight=wtr_seq, epochs=LSTM_EPOCHS, batch_size=32,
              callbacks=[EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)],
              verbose=0)
    return model.predict(Xte_seq, verbose=0).flatten()


def _val_score(proba, y_val, ret_val, metric):
    if metric == 'auc':
        return roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else -np.inf
    r = (proba > 0.5).astype(float) * ret_val
    return r.mean() / r.std() if r.std() > 0 else -np.inf


def tune_model(kind, X_tr, y_tr, w_tr, X_val, y_val, ret_val, metric):
    grid = {'lr': LR_GRID, 'xgb': XGB_GRID, 'lstm': LSTM_GRID}[kind]
    best, best_score = grid[0], -np.inf
    for params in grid:
        try:
            if kind == 'lr':
                p = fit_logistic(X_tr, y_tr, params, w_tr).predict_proba(X_val)[:, 1]
            elif kind == 'xgb':
                p = fit_xgboost(X_tr, y_tr, params, w_tr).predict_proba(X_val)[:, 1]
            else:
                p = fit_predict_lstm(X_tr, y_tr, X_val, params, RANDOM_STATE, w_tr)
            s = _val_score(p, y_val, ret_val, metric)
        except Exception:
            s = -np.inf
        if s > best_score:
            best, best_score = params, s
    return best, best_score


## 7. Pętla ucząca

Pętla zapisuje wyłącznie prawdopodobieństwa przypisane przez modele w oknach testowych. Reguły
inwestycyjne budowane są następnie z tych prawdopodobieństw, co pozwala porównać kilkadziesiąt
wariantów bez ponownego uczenia.


In [ ]:
def prepare_dataset(ticker, target_mode):
    d, feats = feature_data[ticker], FEATURES_BY_TICKER[ticker]
    need = feats + [TARGET_COLS[target_mode], 'next_return', 'next_return_oo', 'next_low_ret',
                    'next_high_ret', 'ann_vol_20', 'w_abs_return', 'Close']
    return d[need].replace([np.inf, -np.inf], np.nan).dropna(), feats


def run_walk_forward(ticker, target_mode, verbose=True):
    dd, feats = prepare_dataset(ticker, target_mode)
    X_all = dd[feats].values.astype(float)
    y_all = dd[TARGET_COLS[target_mode]].values.astype(int)
    dates = dd.index
    purge = max(PURGE, TARGET_HORIZON[target_mode] + 1)

    splits = walk_forward_splits(dates, OOS_START, TEST_WINDOW, STEP, mode=WF_MODE,
                                 rolling_train=ROLLING_TRAIN, purge=purge, embargo=EMBARGO)
    if not splits:
        raise ValueError(f'{ticker}/{target_mode}: brak okien testowych')
    if verbose:
        print(f'  {ticker} / {target_mode}: {len(splits)} okien, {len(dd)} obs., {len(feats)} zmiennych')

    rec = {'proba': {m: [] for m in MODELS}, 'lstm_seeds': [], 'dates': [], 'y_true': [],
           'chosen_params': [], 'selected_features': [], 'fold_id': []}
    last_params = (LR_GRID[0], XGB_GRID[0], LSTM_GRID[0])
    t0 = time.time()

    for k, (tr_idx, te_idx) in enumerate(splits):
        X_tr, y_tr, X_te = X_all[tr_idx], y_all[tr_idx], X_all[te_idx]
        w_tr = dd['w_abs_return'].values[tr_idx] if SAMPLE_WEIGHT_MODE == 'abs_return' else None
        if w_tr is not None:
            w_tr = np.clip(w_tr / (np.mean(w_tr) + 1e-12), 0.1, 10.0)

        keep = prune_correlated(X_tr)
        Xtr, Xte = X_tr[:, keep], X_te[:, keep]

        if DO_TUNING and (k % TUNE_EVERY_N_STEPS == 0):
            in_tr, in_val = inner_validation_split(np.arange(len(tr_idx)), VAL_FRACTION, purge)
            if in_val is not None:
                ret_val = dd['next_return'].values[tr_idx][in_val]
                w_in = w_tr[in_tr] if w_tr is not None else None
                p_lr, _ = tune_model('lr', Xtr[in_tr], y_tr[in_tr], w_in,
                                     Xtr[in_val], y_tr[in_val], ret_val, TUNE_METRIC)
                p_xgb, _ = tune_model('xgb', Xtr[in_tr], y_tr[in_tr], w_in,
                                      Xtr[in_val], y_tr[in_val], ret_val, TUNE_METRIC)
                p_lstm = (tune_model('lstm', Xtr[in_tr], y_tr[in_tr], w_in, Xtr[in_val],
                                     y_tr[in_val], ret_val, TUNE_METRIC)[0]
                          if TUNE_LSTM else LSTM_GRID[0])
            else:
                p_lr, p_xgb, p_lstm = LR_GRID[0], XGB_GRID[0], LSTM_GRID[0]
            last_params = (p_lr, p_xgb, p_lstm)
        else:
            p_lr, p_xgb, p_lstm = last_params

        rec['proba']['LR'].append(fit_logistic(Xtr, y_tr, p_lr, w_tr).predict_proba(Xte)[:, 1])
        rec['proba']['XGB'].append(fit_xgboost(Xtr, y_tr, p_xgb, w_tr).predict_proba(Xte)[:, 1])
        seeds = np.array([fit_predict_lstm(Xtr, y_tr, Xte, p_lstm, RANDOM_STATE + s, w_tr)
                          for s in range(LSTM_N_SEEDS)])
        rec['proba']['LSTM'].append(seeds.mean(axis=0))
        rec['lstm_seeds'].append(seeds)
        rec['dates'].append(dates[te_idx])
        rec['y_true'].append(y_all[te_idx])
        rec['chosen_params'].append(dict(lr=p_lr, xgb=p_xgb, lstm=p_lstm))
        rec['selected_features'].append([feats[i] for i in keep])
        rec['fold_id'].append(np.full(len(te_idx), k))
        if verbose and (k + 1) % 10 == 0:
            print(f'    okno {k+1}/{len(splits)} ({time.time()-t0:.0f}s)')

    idx = pd.DatetimeIndex(np.concatenate([np.array(d) for d in rec['dates']]))
    out = pd.DataFrame(index=idx)
    for m in MODELS:
        out[f'proba_{m}'] = np.concatenate(rec['proba'][m])
    out['y_true'] = np.concatenate(rec['y_true'])
    out['fold'] = np.concatenate(rec['fold_id'])
    out = out[~out.index.duplicated(keep='first')].sort_index()
    out = out.join(dd[['next_return', 'next_return_oo', 'next_low_ret',
                       'next_high_ret', 'ann_vol_20', 'Close']])

    seed_frame = pd.DataFrame(np.concatenate([s.T for s in rec['lstm_seeds']], axis=0), index=idx,
                              columns=[f'lstm_seed_{i}' for i in range(LSTM_N_SEEDS)])
    seed_frame = seed_frame[~seed_frame.index.duplicated(keep='first')].sort_index()
    return dict(oos=out, seeds=seed_frame, params=rec['chosen_params'],
                selected_features=rec['selected_features'], n_splits=len(splits))


## 8. Reguły inwestycyjne

Reguła prosta utrzymuje pozycję, gdy prawdopodobieństwo przekracza 0,5. Konfiguracja główna
dodaje strefę neutralną (wejście powyżej 0,55, wyjście poniżej 0,45), minimalny czas utrzymania
pozycji oraz kontrolę zmienności — wielkość pozycji jest odwrotnie proporcjonalna do bieżącej
zmienności instrumentu.

Próg oznaczony jako dobierany kroczaco jest wyznaczany na podstawie wyłącznie wcześniejszych
okien testowych, bez dostępu do danych przyszłych.


In [ ]:
def _hysteresis_signal(p, entry_thr, exit_thr, min_hold):
    vals = p.values
    ent, ext = np.asarray(entry_thr, float), np.asarray(exit_thr, float)
    out, state, held = np.zeros(len(vals)), 0, 0
    for i, v in enumerate(vals):
        if state == 0:
            if v > ent[i]:
                state, held = 1, 0
        else:
            held += 1
            if v < ext[i] and held >= min_hold:
                state = 0
        out[i] = state
    return pd.Series(out, index=p.index)


def _walk_forward_threshold(oos, proba_col, grid=np.arange(0.40, 0.66, 0.02)):
    folds = sorted(oos['fold'].unique())
    out = pd.Series(0.5, index=oos.index)
    for i, f in enumerate(folds):
        if i == 0:
            continue
        hist = oos[oos['fold'] < f]
        best_t, best_s = 0.5, -np.inf
        for t in grid:
            r = (hist[proba_col] > t).astype(float) * hist['next_return']
            s = r.mean() / r.std() if r.std() > 0 else -np.inf
            if s > best_s:
                best_t, best_s = t, s
        out[oos['fold'] == f] = best_t
    return out


def build_positions(oos, ticker, proba_col, cfg):
    p = oos[proba_col]
    thr = cfg.get('threshold', 0.5)
    thr_series = (_walk_forward_threshold(oos, proba_col) if thr == 'tuned'
                  else pd.Series(float(thr), index=oos.index))
    exit_thr, min_hold = cfg.get('exit_threshold'), int(cfg.get('min_hold', 0))

    if (exit_thr is not None) or (min_hold > 0):
        exit_series = (thr_series if exit_thr is None
                       else pd.Series(float(exit_thr), index=oos.index))
        raw = _hysteresis_signal(p, thr_series.values, exit_series.values, min_hold)
        pos = (raw * ((p - exit_series) * cfg.get('k', 4.0)).clip(0.0, 1.0)
               if cfg.get('mode') == 'scaled' else raw)
    elif cfg.get('mode', 'binary') == 'binary':
        pos = (p > thr_series).astype(float)
    else:
        pos = ((p - thr_series) * cfg.get('k', 4.0)).clip(0.0, 1.0)

    if cfg.get('sma_filter'):
        ratio = feature_data[ticker].get(f"price_sma_{int(cfg['sma_filter'])}_ratio")
        if ratio is not None:
            pos = pos * (ratio.reindex(oos.index).fillna(0) > 0).astype(float)
    if cfg.get('vol_target'):
        scale = cfg['vol_target'] / oos['ann_vol_20'].replace(0, np.nan)
        pos = pos * scale.clip(0.0, MAX_LEVERAGE).fillna(0.0)
    return pos.clip(0.0, MAX_LEVERAGE)


## 9. Symulacja i metryki

Symulacja uwzględnia koszt transakcyjny od każdej zmiany pozycji, oprocentowanie kapitału
nieulokowanego w instrumencie stopą bonów skarbowych oraz ciągłość pozycji między kolejnymi
oknami testowymi.

Wskaźnik CAGR i Calmar są maskowane dla okresów krótszych niż `MIN_YEARS_FOR_CAGR` — annualizacja
z kilku miesięcy daje wartości pozbawione interpretacji.


In [ ]:
def backtest(positions, oos, ticker, cost=TRANSACTION_COST, stop_loss=None, execution=EXECUTION):
    freq = FREQ_MAP[ticker]
    rf = rf_daily_for(oos.index, freq)
    asset_ret = (oos['next_return'] if execution == 'close_to_close' else oos['next_return_oo']).copy()
    if stop_loss is not None and execution == 'close_to_close':
        hit = oos['next_low_ret'] <= -abs(stop_loss)
        asset_ret = asset_ret.where(~hit, -abs(stop_loss) - SLIPPAGE_STOP)
    pos = positions.reindex(oos.index).fillna(0.0)
    turnover = (pos - pos.shift(1).fillna(0.0)).abs()
    strat = pos * asset_ret + (1.0 - pos).clip(lower=0) * rf - turnover * cost
    return strat.rename('ret'), pos, turnover


def buy_and_hold(oos, ticker, cost=TRANSACTION_COST, execution=EXECUTION):
    return backtest(pd.Series(1.0, index=oos.index), oos, ticker, cost=cost, execution=execution)


def cash_only(oos, ticker):
    rf = rf_daily_for(oos.index, FREQ_MAP[ticker])
    return rf.rename('ret'), pd.Series(0.0, index=oos.index), pd.Series(0.0, index=oos.index)


def sma_benchmark(oos, ticker, window=200, cost=TRANSACTION_COST):
    ratio = feature_data[ticker].get(f'price_sma_{window}_ratio')
    pos = (ratio.reindex(oos.index).fillna(0) > 0).astype(float)
    return backtest(pos, oos, ticker, cost=cost)


def stala_ekspozycja(oos, ticker, poziom, cost=TRANSACTION_COST):
    """Strategia pasywna o stalej, obnizonej ekspozycji - punkt odniesienia dla oceny,
    jaka czesc efektu wynika z samego zmniejszenia zaangazowania."""
    return backtest(pd.Series(float(poziom), index=oos.index), oos, ticker, cost=cost)


def compute_metrics(returns, positions=None, turnover=None, ticker=None,
                    asset_returns=None, label=''):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return {}
    freq = FREQ_MAP.get(ticker, 252)
    rf = rf_daily_for(r.index, freq)
    excess = r - rf

    total = (1 + r).prod() - 1
    lata = len(r) / freq
    cagr = (1 + total) ** (1 / lata) - 1 if lata > 0 else np.nan
    if lata < MIN_YEARS_FOR_CAGR:
        cagr = np.nan
    vol = r.std() * np.sqrt(freq)
    sharpe = (excess.mean() * freq) / vol if vol > 1e-9 else np.nan
    dn = excess[excess < 0]
    dvol = dn.std() * np.sqrt(freq) if len(dn) > 1 else np.nan
    eq = (1 + r).cumprod()
    mdd = (eq / eq.cummax() - 1).min()

    out = {'Strategia': label, 'Zwrot': total, 'CAGR': cagr, 'Zmiennosc': vol, 'Sharpe': sharpe,
           'Sortino': (excess.mean() * freq) / dvol if dvol and dvol > 0 else np.nan,
           'Max obsuniecie': mdd,
           'Calmar': cagr / abs(mdd) if (mdd and not np.isnan(cagr)) else np.nan,
           'VaR 95%': r.quantile(0.05), 'Dni': len(r), 'Lata': lata}
    if positions is not None:
        pos = pd.Series(positions).reindex(r.index).fillna(0.0)
        in_mkt = pos > 0
        out['Zaangazowanie'] = pos.mean()
        if asset_returns is not None:
            ar = pd.Series(asset_returns).reindex(r.index)
            out['Skutecznosc'] = (ar[in_mkt] > 0).mean() if in_mkt.any() else np.nan
    if turnover is not None:
        tv = pd.Series(turnover).reindex(r.index).fillna(0.0)
        out['Rotacja'] = tv.sum() / max(lata, 1e-9)
    return out


def vol_matched_benchmark(bench_ret, target_vol, ticker):
    freq = FREQ_MAP[ticker]
    rf = rf_daily_for(bench_ret.index, freq)
    bvol = bench_ret.std() * np.sqrt(freq)
    if bvol <= 0 or not np.isfinite(target_vol):
        return bench_ret
    k = target_vol / bvol
    return k * bench_ret + (1 - k) * rf


def sharpe_ann(r, ticker):
    freq = FREQ_MAP[ticker]
    rf = rf_daily_for(r.index, freq)
    return ((r - rf).mean() * freq) / (r.std() * np.sqrt(freq)) if r.std() > 1e-12 else np.nan


## 9.1. Kroczący dobór parametrów

Zamiast ustalać parametry reguły inwestycyjnej z góry, można wyznaczać je na podstawie
wcześniejszych okien testowych. Dla każdego okna wybierane są konfiguracje, które uzyskały
najlepszy wynik na wszystkich oknach poprzedzających, a pozycja jest średnią ich wskazań.

Uśrednianie wielu konfiguracji zamiast wyboru jednej ogranicza wariancję selekcji: pojedyncza
konfiguracja najlepsza na krótkiej historii bywa wynikiem przypadku, natomiast zbiór kilkunastu
najlepszych odzwierciedla utrzymującą się strukturę. Procedura nie wykorzystuje danych
późniejszych niż okno, dla którego podejmowana jest decyzja.


In [ ]:
def siatka_konfiguracji(entry=(0.52, 0.55, 0.58, 0.60), szerokosc=(0.05, 0.10, 0.15),
                        min_hold=(0, 3, 5, 8), vol_target=(None, 0.20, 0.30, 0.40)):
    out = []
    for e in entry:
        for w in szerokosc:
            for mh in min_hold:
                for vt in vol_target:
                    cfg = dict(mode='binary', threshold=e, exit_threshold=round(e - w, 3),
                               min_hold=mh)
                    if vt is not None:
                        cfg['vol_target'] = vt
                    out.append(cfg)
    return out


def pozycje_kroczace(oos, ticker, proba_col, konfiguracje=None, top_n=10, min_folds=3):
    konfiguracje = konfiguracje if konfiguracje is not None else siatka_konfiguracji()
    rf = rf_daily_for(oos.index, FREQ_MAP[ticker])

    P, R = [], []
    for cfg in konfiguracje:
        pos = build_positions(oos, ticker, proba_col, cfg)
        r, _, _ = backtest(pos, oos, ticker)
        P.append(pos.values)
        R.append((r - rf).values)
    P, R = np.array(P), np.array(R)

    folds = np.asarray(oos['fold'].values)
    out = np.zeros(len(oos))
    for i, f in enumerate(sorted(pd.unique(folds))):
        teraz, hist = folds == f, folds < f
        if i < min_folds or hist.sum() < 60:
            out[teraz] = P[:, teraz].mean(axis=0)
            continue
        m, sd = R[:, hist].mean(axis=1), R[:, hist].std(axis=1)
        sh = np.where(sd > 1e-12, m / sd, -np.inf)
        best = np.argsort(sh)[::-1][:top_n]
        out[teraz] = P[np.ix_(best, teraz)].mean(axis=0)
    return pd.Series(out, index=oos.index).clip(0.0, MAX_LEVERAGE)


## 10. Testy statystyczne


In [ ]:
def pesaran_timmermann(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    T = len(y_true)
    P, Py, Px = np.mean(y_true == y_pred), np.mean(y_true), np.mean(y_pred)
    Pstar = Py * Px + (1 - Py) * (1 - Px)
    var_P = Pstar * (1 - Pstar) / T
    var_Ps = (((2 * Py - 1) ** 2) * Px * (1 - Px) / T + ((2 * Px - 1) ** 2) * Py * (1 - Py) / T
              + 4 * Py * Px * (1 - Py) * (1 - Px) / (T ** 2))
    if var_P - var_Ps <= 0:
        return dict(P=P, P_star=Pstar, stat=np.nan, p_value=np.nan)
    stat = (P - Pstar) / np.sqrt(var_P - var_Ps)
    return dict(P=P, P_star=Pstar, stat=stat, p_value=1 - stats.norm.cdf(stat))


def sharpe_jk_memmel(r1, r2, rf=None):
    r1, r2 = pd.Series(r1).dropna(), pd.Series(r2).dropna()
    idx = r1.index.intersection(r2.index)
    if rf is not None:
        rfx = pd.Series(rf).reindex(idx).fillna(0.0)
        r1, r2 = (r1.loc[idx] - rfx).values, (r2.loc[idx] - rfx).values
    else:
        r1, r2 = r1.loc[idx].values, r2.loc[idx].values
    T = len(r1)
    if T < 30:
        return dict(z=np.nan, p_value=np.nan)
    s1, s2 = r1.std(ddof=1), r2.std(ddof=1)
    if s1 < 1e-9 or s2 < 1e-9:
        return dict(z=np.nan, p_value=np.nan)
    sh1, sh2 = r1.mean() / s1, r2.mean() / s2
    rho = np.corrcoef(r1, r2)[0, 1]
    theta = (1.0 / T) * (2 - 2 * rho + 0.5 * (sh1**2 + sh2**2 - 2 * sh1 * sh2 * rho**2))
    z = (sh1 - sh2) / np.sqrt(theta) if theta > 0 else np.nan
    return dict(z=z, p_value=2 * (1 - stats.norm.cdf(abs(z))) if np.isfinite(z) else np.nan)


def stationary_bootstrap_idx(T, n_boot, mean_block, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    p = 1.0 / max(mean_block, 1)
    idx = np.empty((n_boot, T), dtype=np.int64)
    starts, jumps = rng.integers(0, T, (n_boot, T)), rng.random((n_boot, T)) < p
    for b in range(n_boot):
        i = starts[b, 0]
        for t in range(T):
            idx[b, t] = i
            i = starts[b, t] if jumps[b, t] else (i + 1) % T
    return idx


def bootstrap_ci(returns, freq, rf=None, n_boot=N_BOOTSTRAP, mean_block=MEAN_BLOCK, alpha=0.05):
    r = pd.Series(returns).dropna()
    if rf is not None:
        r = r - pd.Series(rf).reindex(r.index).fillna(0.0)
    r = r.values
    if len(r) < 50:
        return {}
    samp = r[stationary_bootstrap_idx(len(r), n_boot, mean_block)]
    means, stds = samp.mean(axis=1), samp.std(axis=1, ddof=1)
    sh = np.where(stds > 0, means / stds * np.sqrt(freq), np.nan)
    return {'Sharpe': r.mean() / r.std(ddof=1) * np.sqrt(freq),
            'Dolna granica': np.nanpercentile(sh, 100 * alpha / 2),
            'Gorna granica': np.nanpercentile(sh, 100 * (1 - alpha / 2)),
            'P(Sharpe>0)': float(np.nanmean(sh > 0))}


def whites_reality_check(strategy_matrix, benchmark, n_boot=N_BOOTSTRAP, mean_block=MEAN_BLOCK):
    f = strategy_matrix.sub(benchmark, axis=0).dropna()
    T, K = f.shape
    if T < 50 or K == 0:
        return dict(stat=np.nan, p_value=np.nan, K=K)
    fb, fv = f.mean().values, f.values
    stat = np.sqrt(T) * fb.max()
    idx = stationary_bootstrap_idx(T, n_boot, mean_block)
    V = np.array([np.sqrt(T) * np.max(fv[idx[b]].mean(axis=0) - fb) for b in range(n_boot)])
    return dict(stat=stat, p_value=float(np.mean(V > stat)), K=K,
                best=f.columns[int(np.argmax(fb))])


def hansen_spa(strategy_matrix, benchmark, n_boot=N_BOOTSTRAP, mean_block=MEAN_BLOCK):
    f = strategy_matrix.sub(benchmark, axis=0).dropna()
    T, K = f.shape
    if T < 50 or K == 0:
        return dict(stat=np.nan, p_value=np.nan, K=K)
    fv = f.values
    fb = fv.mean(axis=0)
    idx = stationary_bootstrap_idx(T, n_boot, mean_block)
    boot = np.array([fv[idx[b]].mean(axis=0) for b in range(n_boot)])
    w = np.where(boot.std(axis=0) * np.sqrt(T) > 1e-12, boot.std(axis=0) * np.sqrt(T), 1e-12)
    stat = max(0.0, np.max(np.sqrt(T) * fb / w))
    A = w * np.sqrt(2 * np.log(np.log(T)) / T)
    g = np.where(fb >= -A, fb, 0.0)
    Vb = np.maximum(0.0, (np.sqrt(T) * (boot - g) / w).max(axis=1))
    return dict(stat=stat, p_value=float(np.mean(Vb > stat)), K=K)


def deflated_sharpe_ratio(returns, n_trials, freq, trial_variance, rf=None):
    r = pd.Series(returns).dropna()
    if rf is not None:
        r = r - pd.Series(rf).reindex(r.index).fillna(0.0)
    r = r.values
    T = len(r)
    if T < 50 or r.std(ddof=1) < 1e-9:
        return dict(SR=np.nan, SR0=np.nan, DSR=np.nan)
    sr = r.mean() / r.std(ddof=1)
    sk, ku = stats.skew(r), stats.kurtosis(r, fisher=False)
    gamma, N = 0.5772156649, max(int(n_trials), 2)
    sr0 = np.sqrt(trial_variance) * ((1 - gamma) * stats.norm.ppf(1 - 1.0 / N)
                                     + gamma * stats.norm.ppf(1 - 1.0 / (N * np.e)))
    den = np.sqrt(1 - sk * sr + (ku - 1) / 4.0 * sr ** 2)
    dsr = stats.norm.cdf(((sr - sr0) * np.sqrt(T - 1)) / den) if den > 0 else np.nan
    return dict(SR=sr * np.sqrt(freq), SR0=sr0 * np.sqrt(freq), DSR=dsr)


def random_benchmark_test(positions, oos, ticker, n_iter=N_RANDOM, block=20, seed=RANDOM_STATE):
    """Dwa punkty odniesienia pozbawione informacji: sygnal losowy o tej samej sredniej
    ekspozycji oraz permutacja blokowa rzeczywistych decyzji modelu."""
    rng = np.random.default_rng(seed)

    def _sh(arr):
        r, _, _ = backtest(pd.Series(arr, index=oos.index), oos, ticker)
        return sharpe_ann(r, ticker)

    actual, expo, n = _sh(positions.values), float(positions.mean()), len(positions)
    iid = [_sh(rng.binomial(1, min(max(expo, 0), 1), n).astype(float)) for _ in range(n_iter)]
    vals, nb, perm = positions.values, int(np.ceil(n / block)), []
    for _ in range(n_iter):
        order = rng.permutation(nb)
        perm.append(_sh(np.concatenate([vals[i*block:(i+1)*block] for i in order])[:n]))
    return {'Sharpe modelu': actual, 'Zaangazowanie': expo,
            'Sharpe losowy': np.nanmean(iid),
            'Percentyl (losowy)': float(np.nanmean(np.array(iid) < actual) * 100),
            'Sharpe permutacji': np.nanmean(perm),
            'Percentyl (permutacja)': float(np.nanmean(np.array(perm) < actual) * 100)}


def adf_summary(series, name):
    if not OPT['statsmodels']:
        return None
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 100 or s.nunique() < 5:
        return None
    try:
        stat, p, _, nobs, crit, _ = adfuller(s.values, autolag='AIC')
    except Exception:
        return None
    return {'Zmienna': name, 'ADF': stat, 'p-value': p,
            'Wartosc kryt. 5%': crit.get('5%', np.nan), 'Stacjonarna': p < 0.05}


## 11. Analiza portfelowa i ważność zmiennych


In [ ]:
def portfolio_backtest(weights, positions=None, cost=TRANSACTION_COST, freq=252,
                       zadanie='direction_1d'):
    idx = None
    for tk in weights:
        keys = [(t, m) for (t, m) in wf_results if t == tk and m == zadanie]
        if not keys:
            return None
        oos = wf_results[keys[0]]['oos']
        idx = oos.index if idx is None else idx.intersection(oos.index)
    if idx is None or len(idx) < 50:
        return None
    rf = rf_daily_for(idx, freq)
    total, turn = pd.Series(0.0, index=idx), pd.Series(0.0, index=idx)
    invested = pd.Series(0.0, index=idx)
    for tk, w in weights.items():
        k = [(t, m) for (t, m) in wf_results if t == tk and m == zadanie][0]
        oos = wf_results[k]['oos'].reindex(idx)
        pos = (positions[tk].reindex(idx).fillna(0.0) if positions and tk in positions
               else pd.Series(1.0, index=idx))
        eff = pos * w
        total = total + eff * oos['next_return']
        turn = turn + (eff - eff.shift(1).fillna(0.0)).abs()
        invested = invested + eff
    return total + (1.0 - invested).clip(lower=0) * rf - turn * cost, invested, turn


def feature_importance_for(ticker, target_mode, n_repeats=5):
    dd, feats = prepare_dataset(ticker, target_mode)
    X, y = dd[feats].values.astype(float), dd[TARGET_COLS[target_mode]].values.astype(int)
    purge = max(PURGE, TARGET_HORIZON[target_mode] + 1)
    splits = walk_forward_splits(dd.index, OOS_START, TEST_WINDOW, STEP, mode=WF_MODE,
                                 rolling_train=ROLLING_TRAIN, purge=purge, embargo=EMBARGO)
    if not splits:
        return None, None
    tr, te = splits[-1]
    keep = prune_correlated(X[tr])
    names = [feats[i] for i in keep]
    model = fit_xgboost(X[tr][:, keep], y[tr], XGB_GRID[0])
    gain = pd.Series(model.feature_importances_, index=names).sort_values(ascending=False)
    try:
        pi = permutation_importance(model, X[te][:, keep], y[te], n_repeats=n_repeats,
                                    random_state=RANDOM_STATE, scoring='roc_auc')
        perm = pd.Series(pi.importances_mean, index=names).sort_values(ascending=False)
    except Exception:
        perm = pd.Series(dtype=float)
    return gain, perm


def grupa_zmiennej(n):
    if n.startswith('x_'):
        return 'rynkowe i makroekonomiczne'
    if n.startswith('c_'):
        return 'sieciowe i sentyment'
    return 'techniczne'


## 12. Przygotowanie danych

Funkcja `przygotuj_dane()` pobiera notowania, szeregi zewnętrzne i stopę wolną od ryzyka,
konstruuje zmienne objaśniające i objaśniane dla wszystkich instrumentów oraz udostępnia je
w przestrzeni globalnej. Zwraca zestawienie liczby zmiennych i obserwacji.


In [ ]:
def przygotuj_dane(verbose=True):
    raw_data = {}
    for tk in list(TICKERS) + [TICKER_WERYFIKACYJNY]:
        raw_data[tk] = download_yf(tk)

    exog_raw = {}
    for tk, spec in EXOG_SPEC.items():
        try:
            exog_raw[tk] = download_yf(tk)['Close'].rename(spec['name'])
        except Exception:
            pass
    exog_levels = pd.DataFrame(exog_raw.values()).T if exog_raw else pd.DataFrame()
    if not exog_levels.empty:
        exog_levels.columns = [EXOG_SPEC[k]['name'] for k in exog_raw]
    exog_kinds = {EXOG_SPEC[k]['name']: EXOG_SPEC[k]['kind'] for k in exog_raw}
    exog_source = {EXOG_SPEC[k]['name']: k for k in exog_raw}

    try:
        rf_annual = (download_yf('^IRX')['Close'] / 100.0).clip(lower=0.0)
    except Exception:
        rf_annual = pd.Series(dtype=float)

    crypto_extra = {}
    for label, fn in [('FearGreed', fetch_fear_greed), ('OnChain', fetch_onchain),
                      ('Funding', lambda: fetch_funding('BTCUSDT')),
                      ('FundingETH', lambda: fetch_funding('ETHUSDT'))]:
        try:
            crypto_extra[label] = fn()
        except Exception:
            pass

    globals().update(dict(raw_data=raw_data, exog_levels=exog_levels, exog_kinds=exog_kinds,
                          exog_source=exog_source, rf_annual=rf_annual,
                          crypto_extra=crypto_extra))

    feature_data, FEATURES_BY_TICKER, zestawienie = {}, {}, []
    for tk in list(TICKERS) + [TICKER_WERYFIKACYJNY]:
        d = add_technical_features(raw_data[tk], FREQ_MAP[tk])
        d = d.join(build_exog_features(d.index, exclude_tickers=(tk,)))
        if tk == 'BTC-USD':
            d = d.join(build_crypto_features(d.index))
        elif tk == TICKER_WERYFIKACYJNY:
            d = d.join(build_eth_features(d.index))
        d = add_targets(d, FREQ_MAP[tk]).dropna(axis=1, how='all')

        feats = [c for c in d.columns if c not in NON_FEATURE_COLS]
        feature_data[tk], FEATURES_BY_TICKER[tk] = d, feats
        dv = d.dropna(subset=feats)
        zestawienie.append({
            'Instrument': TICKERS.get(tk, 'Ethereum'),
            'Techniczne': len([c for c in feats if not c.startswith(('x_', 'c_'))]),
            'Rynkowe i makro': len([c for c in feats if c.startswith('x_')]),
            'Sieciowe i sentyment': len([c for c in feats if c.startswith('c_')]),
            'Razem': len(feats), 'Obserwacje': len(dv), 'Od': dv.index.min().date(),
        })

    globals().update(dict(feature_data=feature_data, FEATURES_BY_TICKER=FEATURES_BY_TICKER))
    if verbose and not rf_annual.empty:
        print(f'Srednia stopa wolna od ryzyka w okresie: {rf_annual.mean():.2%}')
    return pd.DataFrame(zestawienie).set_index('Instrument')


def wczytaj_cache():
    """Wczytuje zapisane wyniki petli uczacej. Zwraca slownik z czterema zestawami."""
    out = {}
    for nazwa, sciezka in SCIEZKI_CACHE.items():
        if os.path.exists(sciezka):
            with open(sciezka, 'rb') as f:
                out[nazwa] = pickle.load(f)
        else:
            out[nazwa] = {}
    return out
